In [1]:
import sys, os, json, shutil
import numpy as np
import pandas as pd
import torch
import networkx as nx
from tqdm import tqdm

BASE = "/kaggle/working/phase2_recovery"
CKPT_DIR = f"{BASE}/checkpoints"
CACHE_DIR = f"{BASE}/cache"
SRC_DIR = f"{BASE}/source_code"
OUT_DIR = f"{BASE}/outputs/best_checkpoint_baseline"
CACHE_PATH = os.path.join(CACHE_DIR, "phase2_cache")

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "predictions"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "meshes"), exist_ok=True)

shutil.copy2(
    "/kaggle/input/datasets/risha8750/phase2-core3/phase2_core.py",
    os.path.join(SRC_DIR, "phase2_core.py"),
)
for fname in ["corner_best.pt", "edge_best.pt"]:
    shutil.copy2(
        f"/kaggle/input/datasets/risha8750/latest-models/Latest_models/{fname}",
        os.path.join(CKPT_DIR, fname),
    )
shutil.copytree(
    "/kaggle/input/notebooks/risha8750/major-project-phase-2a-cache/phase2_cache",
    CACHE_PATH,
    dirs_exist_ok=True,
)

sys.path.insert(0, SRC_DIR)
import phase2_core

device = "cuda" if torch.cuda.is_available() else "cpu"
phase2_core.seed_everything(42)
print("Device:", device)

Device: cuda


In [2]:
corner_ckpt = torch.load(f"{CKPT_DIR}/corner_best.pt", map_location=device, weights_only=False)
edge_ckpt = torch.load(f"{CKPT_DIR}/edge_best.pt", map_location=device, weights_only=False)

corner_model = phase2_core.CornerPointNet2()
corner_model.load_state_dict(corner_ckpt["model"])
corner_model = corner_model.to(device).eval()
print("Corner model epoch:", corner_ckpt.get("epoch"))

edge_model = phase2_core.EdgeGraphNet()
edge_model.load_state_dict(edge_ckpt["model"])
edge_model = edge_model.to(device).eval()
print("Edge model epoch:", edge_ckpt.get("epoch"))

Corner model epoch: 19
Edge model epoch: 7


In [3]:
def order_footprint(corners_xy, edges):
    n = len(corners_xy)
    graph = nx.Graph()
    graph.add_nodes_from(range(n))
    graph.add_edges_from([tuple(map(int, e)) for e in edges])

    degrees = dict(graph.degree())
    is_clean_cycle = (
        graph.number_of_edges() == n
        and all(d == 2 for d in degrees.values())
        and nx.is_connected(graph)
    )

    if is_clean_cycle:
        cycle = list(nx.cycle_basis(graph)[0])
        if len(cycle) == n:
            return cycle

    centroid = corners_xy.mean(axis=0)
    angles = np.arctan2(corners_xy[:, 1] - centroid[1], corners_xy[:, 0] - centroid[0])
    return list(np.argsort(angles))


def fan_triangulate(ordered_indices):
    faces = []
    for i in range(1, len(ordered_indices) - 1):
        faces.append((ordered_indices[0], ordered_indices[i], ordered_indices[i + 1]))
    return faces


def build_extruded_mesh(corners, edges, wall_height=0.3):
    corners = np.asarray(corners, dtype=np.float64)
    xy = corners[:, :2]

    ordered = order_footprint(xy, edges)
    top = corners[ordered]
    n = len(top)

    floor_z = top[:, 2].min() - wall_height
    bottom = top.copy()
    bottom[:, 2] = floor_z

    vertices = np.vstack([top, bottom])
    vertices[:, 2] -= floor_z  # floor sits exactly at Z = 0 (before axis swap)

    faces = []
    faces.extend(fan_triangulate(list(range(n))))
    for a, b, c in fan_triangulate(list(range(n))):
        faces.append((c + n, b + n, a + n))
    for i in range(n):
        j = (i + 1) % n
        faces.append((i, j, n + j))
        faces.append((i, n + j, n + i))

    return vertices, np.asarray(faces, dtype=np.int64)


def make_double_sided(faces):
    doubled = []
    for f in faces:
        doubled.append(f)
        doubled.append((f[0], f[2], f[1]))
    return doubled


def write_obj(vertices, faces, path):
    with open(path, "w") as f:
        for v in vertices:
            x, y, z = v
            f.write(f"v {x} {z} {y}\n")  # Y/Z swapped so floor sits at Y=0 for viewers
        for face in faces:
            f.write(f"f {face[0]+1} {face[1]+1} {face[2]+1}\n")

In [4]:
test_ds = phase2_core.RoofDataset(cache_root=CACHE_PATH, split="test", augment=False)

rows = []
edge_threshold = 0.5

for i in tqdm(range(len(test_ds))):
    sample = test_ds[i]
    building_id = int(sample["building_id"])
    points = sample["points"].unsqueeze(0).to(device)

    with torch.no_grad():
        logits, offsets, features = corner_model(points)

    scale_val = float(sample["scale"])
    pred_coords, pred_scores = phase2_core.predict_corners(
        points[0], logits[0], offsets[0], scale_val
    )

    n_true = int(sample["corner_count"])
    truth_corners = sample["corners"][:n_true].numpy()
    n_true_edges = int(sample["edge_count"])
    truth_edges = sample["edges"][:n_true_edges].numpy()

    if pred_coords.shape[0] < 2:
        pred_edges = np.zeros((0, 2), dtype=np.int64)
    else:
        points_np = points[0][:, :3].cpu().numpy()
        features_np = features[0].cpu().numpy()
        dists = np.linalg.norm(points_np[None, :, :] - pred_coords[:, None, :], axis=-1)
        nearest_idx = dists.argmin(axis=1)
        corner_features_np = features_np[nearest_idx]

        corner_xyz_t = torch.from_numpy(pred_coords).float().to(device)
        corner_scores_t = torch.from_numpy(pred_scores).float().to(device)
        corner_features_t = torch.from_numpy(corner_features_np).float().to(device)

        with torch.no_grad():
            edge_pairs, edge_logits = edge_model(corner_xyz_t, corner_scores_t, corner_features_t)

        edge_probs = torch.sigmoid(edge_logits).cpu().numpy()
        edge_pairs_np = edge_pairs.cpu().numpy()
        pred_edges = edge_pairs_np[edge_probs > edge_threshold]

    counts = phase2_core.reconstruction_counts(
        pred_coords, pred_edges, truth_corners, truth_edges, tolerance_m=0.5
    )
    counts["building_id"] = building_id
    rows.append(counts)

    np.savez(
        os.path.join(OUT_DIR, "predictions", f"{building_id}.npz"),
        corners=pred_coords, corner_scores=pred_scores, edges=pred_edges,
    )

df = pd.DataFrame(rows)[["building_id","corner_tp","corner_pred","corner_true","corner_offset_sum","edge_tp","edge_pred","edge_true"]]
df.to_csv(os.path.join(OUT_DIR, "test_per_building.csv"), index=False)
print("Done:", len(df), "rows")

100%|██████████| 3235/3235 [05:06<00:00, 10.57it/s]

Done: 3235 rows


In [5]:
perfect = df[(df["corner_tp"]==df["corner_pred"]) & (df["corner_pred"]==df["corner_true"]) &
             (df["edge_tp"]==df["edge_pred"]) & (df["edge_pred"]==df["edge_true"])]

print("Total perfect reconstructions:", len(perfect))

eight_corner = perfect[perfect["corner_true"]==8]
six_corner = perfect[perfect["corner_true"]==6].head(8)
five_corner = perfect[perfect["corner_true"]==5]
four_corner = perfect[perfect["corner_true"]==4].head(8)

showcase = pd.concat([eight_corner, six_corner, five_corner, four_corner]).drop_duplicates()
showcase_ids = showcase["building_id"].tolist()
all_perfect_ids = perfect["building_id"].tolist()

print("Showcase set size:", len(showcase_ids))
print("All perfect count:", len(all_perfect_ids))

Total perfect reconstructions: 165
Showcase set size: 19
All perfect count: 165


In [6]:
mesh_index_rows = []
failed_all = []

for bid in all_perfect_ids:
    try:
        npz_path = os.path.join(OUT_DIR, "predictions", f"{bid}.npz")
        data = np.load(npz_path)
        pred_coords_b = data["corners"]
        pred_edges_b = data["edges"]

        vertices, faces = build_extruded_mesh(pred_coords_b, pred_edges_b, wall_height=0.3)
        faces = make_double_sided(faces)

        mesh_dir = os.path.join(OUT_DIR, "meshes", str(bid))
        os.makedirs(mesh_dir, exist_ok=True)
        obj_path = os.path.join(mesh_dir, "building_mesh.obj")

        write_obj(vertices, faces, obj_path)

        mesh_index_rows.append({
            "building_id": bid, "num_corners": len(pred_coords_b),
            "num_faces": len(faces), "obj_path": obj_path, "status": "ok"
        })
    except Exception as e:
        failed_all.append((bid, str(e)))
        mesh_index_rows.append({
            "building_id": bid, "num_corners": 0, "num_faces": 0,
            "obj_path": None, "status": f"failed: {e}"
        })

mesh_index_df = pd.DataFrame(mesh_index_rows)
print(f"Exported: {(mesh_index_df['status']=='ok').sum()}/{len(all_perfect_ids)}")
if failed_all:
    print("Failed:", failed_all[:10])

Exported: 165/165


In [7]:
ok_rows = mesh_index_df[mesh_index_df["status"]=="ok"]
bad = []
for _, row in ok_rows.iterrows():
    with open(row["obj_path"]) as f:
        ys = [float(l.split()[2]) for l in f if l.startswith("v ")]
    if abs(min(ys)) > 1e-4:
        bad.append((row["building_id"], min(ys)))

print(f"{len(ok_rows)-len(bad)}/{len(ok_rows)} buildings confirmed flat on ground (Y=0)")
if bad:
    print("Broken:", bad[:10])

165/165 buildings confirmed flat on ground (Y=0)


In [8]:
zip_path = "/kaggle/working/fresh_meshes"
shutil.make_archive(zip_path, "zip", os.path.join(OUT_DIR, "meshes"))
mb = os.path.getsize(zip_path + ".zip") / (1024*1024)
print(f"Created {zip_path}.zip ({mb:.1f} MB)")

Created /kaggle/working/fresh_meshes.zip (0.1 MB)


In [9]:
verification_rows = []

for bid in showcase_ids[:20]:
    obj_path = os.path.join(OUT_DIR, "meshes", str(bid), "building_mesh.obj")
    if not os.path.exists(obj_path):
        continue

    zs = []
    with open(obj_path) as f:
        for line in f:
            if line.startswith("v "):
                z = float(line.split()[3])
                zs.append(z)

    min_z = min(zs)
    max_z = max(zs)

    verification_rows.append({
        "building_id": bid,
        "min_z": round(min_z, 6),
        "max_z": round(max_z, 6),
        "floor_touches_zero": abs(min_z) < 1e-4,
    })

verify_df = pd.DataFrame(verification_rows)
print(verify_df.to_string(index=False))

all_pass = verify_df["floor_touches_zero"].all()
print(f"\nALL {len(verify_df)} BUILDINGS HAVE FLOOR AT Z=0: {all_pass}")

 building_id     min_z    max_z  floor_touches_zero
        6088 -0.542487 0.558760               False
         547 -0.971144 0.921982               False
        1616 -0.845597 0.868951               False
        2765 -0.916328 0.918736               False
        2846 -0.687026 0.609412               False
        3737 -0.736889 0.722597               False
        4235 -0.967649 0.977471               False
        4622 -0.687825 0.667348               False
        4722 -0.695047 0.712852               False
       11369 -0.920699 0.930000               False
       22116 -0.896447 0.921905               False
          70 -0.951209 0.954725               False
         266 -0.997069 0.914442               False
         412 -0.962735 0.954041               False
         434 -0.891483 0.897361               False
         618 -0.972229 0.975396               False
         862 -0.357269 0.317417               False
         990 -0.852073 0.957547               False
        1456

In [10]:
import pandas as pd
import numpy as np

for name, val in list(globals().items()):
    if isinstance(val, (pd.DataFrame, np.ndarray, list, dict)):
        try:
            size = len(val)
        except:
            size = "?"
        print(f"{name}: {type(val).__name__}, length={size}")

_ih: list, length=11
_oh: dict, length=0
_dh: list, length=1
In: list, length=11
Out: dict, length=0
corner_ckpt: dict, length=6
edge_ckpt: dict, length=6
rows: list, length=3235
sample: dict, length=8
pred_coords: ndarray, length=10
pred_scores: ndarray, length=10
truth_corners: ndarray, length=22
truth_edges: ndarray, length=31
points_np: ndarray, length=2048
features_np: ndarray, length=2048
dists: ndarray, length=10
nearest_idx: ndarray, length=10
corner_features_np: ndarray, length=10
edge_probs: ndarray, length=45
edge_pairs_np: ndarray, length=45
pred_edges: ndarray, length=15
counts: dict, length=8
df: DataFrame, length=3235
perfect: DataFrame, length=165
eight_corner: DataFrame, length=1
six_corner: DataFrame, length=8
five_corner: DataFrame, length=2
four_corner: DataFrame, length=8
showcase: DataFrame, length=19
showcase_ids: list, length=19
all_perfect_ids: list, length=165
mesh_index_rows: list, length=165
failed_all: list, length=0
pred_coords_b: ndarray, length=4
pred_ed

In [11]:
print(rows[0])
print()
print(df.columns.tolist())
print()
print(mesh_index_df.columns.tolist())
print(mesh_index_df.head())

{'corner_tp': 4, 'corner_pred': 4, 'corner_true': 12, 'corner_offset_sum': 0.21195781230926514, 'edge_tp': 2, 'edge_pred': 3, 'edge_true': 19, 'building_id': 5}

['building_id', 'corner_tp', 'corner_pred', 'corner_true', 'corner_offset_sum', 'edge_tp', 'edge_pred', 'edge_true']

['building_id', 'num_corners', 'num_faces', 'obj_path', 'status']
   building_id  num_corners  num_faces  \
0           70            4         24   
1          266            4         24   
2          412            4         24   
3          434            4         24   
4          547            6         40   

                                            obj_path status  
0  /kaggle/working/phase2_recovery/outputs/best_c...     ok  
1  /kaggle/working/phase2_recovery/outputs/best_c...     ok  
2  /kaggle/working/phase2_recovery/outputs/best_c...     ok  
3  /kaggle/working/phase2_recovery/outputs/best_c...     ok  
4  /kaggle/working/phase2_recovery/outputs/best_c...     ok  


In [12]:
perfect = df[(df["corner_tp"]==df["corner_pred"]) & (df["corner_pred"]==df["corner_true"]) &
             (df["edge_tp"]==df["edge_pred"]) & (df["edge_pred"]==df["edge_true"])]

eight_corner = perfect[perfect["corner_true"]==8]
six_corner = perfect[perfect["corner_true"]==6].head(8)
five_corner = perfect[perfect["corner_true"]==5]
four_corner = perfect[perfect["corner_true"]==4].head(8)
showcase = pd.concat([eight_corner, six_corner, five_corner, four_corner]).drop_duplicates()
showcase_ids = showcase["building_id"].tolist()

all_perfect_ids = perfect["building_id"].tolist()

mesh_index_rows = []
failed_all = []

for bid in all_perfect_ids:
    try:
        npz_path = os.path.join(OUT_DIR, "predictions", f"{bid}.npz")
        data = np.load(npz_path)
        pred_coords_b = data["corners"]
        pred_edges_b = data["edges"]

        vertices, faces = build_extruded_mesh(pred_coords_b, pred_edges_b, wall_height=0.3)
        faces = make_double_sided(faces)
        mesh_dir = os.path.join(OUT_DIR, "meshes", str(bid))
        os.makedirs(mesh_dir, exist_ok=True)
        obj_path = os.path.join(mesh_dir, "building_mesh.obj")

        write_obj(vertices, faces, obj_path)   # <-- local function, correct arg order, has Y/Z swap

        mesh_index_rows.append({
            "building_id": bid, "num_corners": len(pred_coords_b),
            "num_faces": len(faces), "obj_path": obj_path, "status": "ok"
        })
    except Exception as e:
        failed_all.append((bid, str(e)))
        mesh_index_rows.append({
            "building_id": bid, "num_corners": 0, "num_faces": 0,
            "obj_path": None, "status": f"failed: {e}"
        })

mesh_index_df = pd.DataFrame(mesh_index_rows)
print(f"Exported: {(mesh_index_df['status']=='ok').sum()}/{len(all_perfect_ids)}")
if failed_all:
    print("Failed:", failed_all[:10])

Exported: 165/165


In [13]:
ok_rows = mesh_index_df[mesh_index_df["status"]=="ok"]
bad = []
for _, row in ok_rows.iterrows():
    with open(row["obj_path"]) as f:
        ys = [float(l.split()[2]) for l in f if l.startswith("v ")]
    if abs(min(ys)) > 1e-4:
        bad.append((row["building_id"], min(ys)))

print(f"{len(ok_rows)-len(bad)}/{len(ok_rows)} buildings confirmed flat on ground (Y=0)")
if bad:
    print("Broken:", bad[:10])

165/165 buildings confirmed flat on ground (Y=0)


In [14]:
import inspect
print(inspect.getsource(write_obj))

def write_obj(vertices, faces, path):
    with open(path, "w") as f:
        for v in vertices:
            x, y, z = v
            f.write(f"v {x} {z} {y}\n")  # Y/Z swapped so floor sits at Y=0 for viewers
        for face in faces:
            f.write(f"f {face[0]+1} {face[1]+1} {face[2]+1}\n")



In [15]:
with open(mesh_index_df[mesh_index_df["building_id"]==70]["obj_path"].iloc[0]) as f:
    print(f.read())

v 0.2804241180419922 0.3 -0.951208770275116
v 0.8669655919075012 0.3360189815983176 -0.49910011887550354
v -0.2877824902534485 0.3220209639519453 0.9547248482704163
v -0.8531683087348938 0.34093723744153975 0.46261176466941833
v 0.2804241180419922 0.0 -0.951208770275116
v 0.8669655919075012 0.0 -0.49910011887550354
v -0.2877824902534485 0.0 0.9547248482704163
v -0.8531683087348938 0.0 0.46261176466941833
f 1 2 3
f 1 3 2
f 1 3 4
f 1 4 3
f 7 6 5
f 7 5 6
f 8 7 5
f 8 5 7
f 1 2 6
f 1 6 2
f 1 6 5
f 1 5 6
f 2 3 7
f 2 7 3
f 2 7 6
f 2 6 7
f 3 4 8
f 3 8 4
f 3 8 7
f 3 7 8
f 4 1 5
f 4 5 1
f 4 5 8
f 4 8 5



In [16]:
import time
print("Current time:", time.strftime("%H:%M:%S"))
print()
print(mesh_index_df["status"].value_counts())
print()
sample_path = mesh_index_df[mesh_index_df["building_id"]==70]["obj_path"].iloc[0]
print("Path:", sample_path)
with open(sample_path) as f:
    print(f.read())

Current time: 17:55:39

status
ok    165
Name: count, dtype: int64

Path: /kaggle/working/phase2_recovery/outputs/best_checkpoint_baseline/meshes/70/building_mesh.obj
v 0.2804241180419922 0.3 -0.951208770275116
v 0.8669655919075012 0.3360189815983176 -0.49910011887550354
v -0.2877824902534485 0.3220209639519453 0.9547248482704163
v -0.8531683087348938 0.34093723744153975 0.46261176466941833
v 0.2804241180419922 0.0 -0.951208770275116
v 0.8669655919075012 0.0 -0.49910011887550354
v -0.2877824902534485 0.0 0.9547248482704163
v -0.8531683087348938 0.0 0.46261176466941833
f 1 2 3
f 1 3 2
f 1 3 4
f 1 4 3
f 7 6 5
f 7 5 6
f 8 7 5
f 8 5 7
f 1 2 6
f 1 6 2
f 1 6 5
f 1 5 6
f 2 3 7
f 2 7 3
f 2 7 6
f 2 6 7
f 3 4 8
f 3 8 4
f 3 8 7
f 3 7 8
f 4 1 5
f 4 5 1
f 4 5 8
f 4 8 5



In [17]:
import shutil
shutil.make_archive("/kaggle/working/fresh_meshes", "zip", "/kaggle/working/phase2_recovery/outputs/best_checkpoint_baseline/meshes")
print("Zip created at /kaggle/working/fresh_meshes.zip")

Zip created at /kaggle/working/fresh_meshes.zip


In [18]:
def shape_score(building_id, output_dir):
    npz_path = os.path.join(output_dir, "predictions", f"{building_id}.npz")
    data = np.load(npz_path)
    coords = data["corners"][:, :2]
    n = len(coords)
    if n < 5:
        return None  # 4-corner = flat box, not a gabled shape

    angles = []
    for i in range(n):
        p_prev, p_curr, p_next = coords[i-1], coords[i], coords[(i+1)%n]
        v1, v2 = p_prev - p_curr, p_next - p_curr
        cosang = np.dot(v1,v2)/(np.linalg.norm(v1)*np.linalg.norm(v2))
        ang = np.degrees(np.arccos(np.clip(cosang,-1,1)))
        angles.append(ang)

    angles = np.array(angles)
    min_angle = angles.min()
    max_angle = angles.max()
    is_clean = (min_angle > 30) and (max_angle < 150)
    return {
        "building_id": building_id,
        "n_corners": n,
        "min_angle": round(min_angle,1),
        "max_angle": round(max_angle,1),
        "clean_gable_shape": is_clean
    }

results = [shape_score(bid, OUT_DIR) for bid in all_perfect_ids]
results = [r for r in results if r is not None]
shape_df = pd.DataFrame(results)

good_houses = shape_df[shape_df["clean_gable_shape"]].sort_values("n_corners", ascending=False)
print(good_houses.to_string(index=False))

 building_id  n_corners  min_angle  max_angle  clean_gable_shape
        2846          6  31.400000  95.300003               True
       15828          6  35.000000  91.099998               True
       16519          6  37.200001  96.500000               True
       33249          6  39.099998 131.199997               True
       40473          6  41.700001  89.800003               True
       40006          6  42.400002  92.699997               True
       40874          6  30.200001 131.300003               True
       22116          5  45.000000  89.500000               True
